# streamrelay: Zero to Hero Tutorial

**streamrelay** solves a fundamental gap in HPC job schedulers: they run a function and return a single result. This library adds a lightweight bidirectional channel so your HPC function's output streams out **in real time**, token by token.

Works with any incrementally produced output: LLM tokens, simulation checkpoints, solver convergence metrics, processed records, or any JSON-serializable payload.

| Part | What you'll learn | Time |
|------|-------------------|------|
| **1. Concepts** | Why a relay? The channel model | 5 min |
| **2. Local relay** | Start a relay, send/receive | 10 min |
| **3. Synchronous API** | RelayProducer + RelayConsumer | 10 min |
| **4. Async API** | FastAPI SSE endpoint | 10 min |
| **5. Encryption** | AES-256-GCM end-to-end | 5 min |
| **6. Globus Compute** | StreamingExecutor | 10 min |
| **7. Production** | Lifecycle CLI, systemd, Caddy | 10 min |

## Prerequisites

```bash
pip install streamrelay                # core: server + producer + consumer
pip install streamrelay[globus]        # add: globus-compute-sdk
```

---
## Part 1: Why a relay?

### The firewall problem

HPC compute nodes live behind strict institutional firewalls:
- They can make **outbound** connections (how Globus Compute, Slurm, etc. work).
- They **reject all inbound** connections — no SSH, no HTTP, no WebSocket from outside.

Your web app may also be behind a NAT or corporate firewall with no inbound ports.

### The relay solution

Put a small relay server on a publicly reachable machine (a $5/month VM works). Both sides connect **outbound** to it:

```
HPC node  ──outbound──►  relay  ◄──outbound──  your app
                            │
                       forwards messages in real time
```

This is the same principle used by TURN servers in WebRTC video calls.

### The channel model

A **channel** is a matched pair identified by a random UUID:
- Producer connects to `/produce/{channel_id}` — sends tokens
- Consumer connects to `/consume/{channel_id}` — receives tokens

The relay matches them by ID and forwards all messages. UUIDs have 122 bits of entropy — impossible to guess, so channel isolation is guaranteed.

### Message protocol

All messages are JSON. The relay forwards them unchanged:
```json
{"type": "token",  "content": "Hello"}   ← one piece of output
{"type": "done",   "usage": {...}}        ← generation complete
{"type": "error",  "message": "..."}      ← something went wrong
```

When encryption is enabled:
```json
{"type": "enc", "d": "<base64(nonce + ciphertext + GCM tag)>"}
```

---
## Part 2: Start a local relay

In [ ]:
import subprocess, sys, time

# Start the relay server as a background subprocess (simulates a VM)
relay_proc = subprocess.Popen(
    [sys.executable, "-m", "streamrelay.server",
     "--port", "8765",
     "--log-level", "WARNING"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(1)

relay_url = "ws://localhost:8765"
print(f"Relay running at: {relay_url}")

In [ ]:
# Verify it's healthy via WebSocket
from websockets.sync.client import connect as ws_connect
import json

with ws_connect(f"{relay_url}/health") as ws:
    health = json.loads(ws.recv())

print(f"Status: {health['status']}")
print(f"Active channels: {health['active_channels']}")
print(f"Timestamp: {health['timestamp']}")

---
## Part 3: Synchronous API — RelayProducer and RelayConsumer

### 3.1 Basic send/receive

In [ ]:
import uuid, threading, time
from streamrelay import RelayProducer, RelayConsumer

# Both sides share the same channel_id
channel_id = str(uuid.uuid4())
print(f"Channel: {channel_id[:8]}...")

# Producer side (simulates an HPC function)
def produce():
    """Use as a context manager — 'done' is sent automatically on exit."""
    with RelayProducer(relay_url, channel_id) as relay:
        for word in "The quick brown fox".split():
            relay.send_token(word + " ")
            time.sleep(0.1)

# Start the producer in a background thread
threading.Thread(target=produce, daemon=True).start()

# Consumer side (your application)
print("Receiving: ", end="")
for token in RelayConsumer(relay_url, channel_id).stream():
    print(token, end="", flush=True)
print("\nDone!")

### 3.2 Timing: consumer connects BEFORE or AFTER producer

The relay buffers messages when the consumer isn't connected yet. Both orderings work.

In [ ]:
# Ordering 1: Producer starts FIRST (common with Globus Compute)
# Tokens are buffered until consumer connects

ch1 = str(uuid.uuid4())

def fast_producer():
    with RelayProducer(relay_url, ch1) as relay:
        for i in range(5):
            relay.send_token(f"early_token_{i} ")
    print("[producer] done")

# Producer finishes BEFORE consumer even connects
threading.Thread(target=fast_producer, daemon=True).start()
time.sleep(0.5)  # producer has time to finish

# Consumer arrives late — gets all buffered tokens
result = RelayConsumer(relay_url, ch1).collect()
print(f"[consumer] got: {result!r}")

In [ ]:
# Ordering 2: Consumer connects FIRST (common for web apps)
# Consumer waits for producer, no buffering needed

ch2 = str(uuid.uuid4())
tokens_received = []

def consumer_thread():
    for t in RelayConsumer(relay_url, ch2).stream():
        tokens_received.append(t)

# Consumer starts first
t = threading.Thread(target=consumer_thread, daemon=True)
t.start()

time.sleep(0.2)  # consumer is waiting

# Producer connects after a delay
with RelayProducer(relay_url, ch2) as relay:
    relay.send_token("late_arrival ")

t.join(timeout=3)
print(f"Received: {tokens_received}")

### 3.3 Sending structured data (not just text tokens)

In [ ]:
import json

ch3 = str(uuid.uuid4())

def produce_metrics():
    """Send structured JSON as token content."""
    with RelayProducer(relay_url, ch3) as relay:
        for step in range(5):
            metric = {"step": step, "loss": 1.0 / (step + 1), "acc": step / 5}
            relay.send_token(json.dumps(metric))  # serialize to string, send as token
            time.sleep(0.05)

threading.Thread(target=produce_metrics, daemon=True).start()

print("Training metrics:")
for raw_token in RelayConsumer(relay_url, ch3).stream():
    m = json.loads(raw_token)  # parse on consumer side
    print(f"  step={m['step']:2d}  loss={m['loss']:.4f}  acc={m['acc']:.2f}")

### 3.4 Error propagation

In [ ]:
ch4 = str(uuid.uuid4())

def produce_error():
    with RelayProducer(relay_url, ch4) as relay:
        relay.send_token("starting computation... ")
        time.sleep(0.1)
        raise ValueError("GPU out of memory on node ghi2-002")
    # __exit__ catches the exception and sends:
    # {"type": "error", "message": "ValueError: GPU out of memory..."}
    # {"type": "done"}

threading.Thread(target=produce_error, daemon=True).start()

try:
    for token in RelayConsumer(relay_url, ch4).stream():
        print(f"  token: {token!r}")
except RuntimeError as e:
    print(f"Caught: {e}")

### 3.5 Usage statistics via send_done()

In [ ]:
# For LLM use cases: include token count in the 'done' message
ch5 = str(uuid.uuid4())

def produce_with_usage():
    with RelayProducer(relay_url, ch5) as relay:
        words = "Hello world from the GPU".split()
        for w in words:
            relay.send_token(w + " ")
        # Override the automatic done with usage stats
        relay.send_done(usage={"prompt_tokens": 5, "completion_tokens": len(words)})
    # Note: calling send_done() then letting __exit__ send done again is fine —
    # but better practice is to call send_done() explicitly and NOT use context manager

threading.Thread(target=produce_with_usage, daemon=True).start()

# To access the 'done' message's usage field, use the low-level WebSocket directly:
from websockets.sync.client import connect as ws_connect

tokens = []
usage = {}

with ws_connect(f"{relay_url}/consume/{ch5}") as ws:
    for raw in ws:
        msg = json.loads(raw)
        if msg["type"] == "token":
            tokens.append(msg["content"])
        elif msg["type"] == "done":
            usage = msg.get("usage", {})
            break

print(f"Response: {''.join(tokens)}")
print(f"Usage: {usage}")

---
## Part 4: Async API — for FastAPI and asyncio applications

### 4.1 Async consumer

In [ ]:
ch6 = str(uuid.uuid4())

# Synchronous producer (still sync — simulates HPC side which runs in a thread)
def sync_produce():
    with RelayProducer(relay_url, ch6) as relay:
        for i in range(4):
            relay.send_token(f"async_chunk_{i} ")
            time.sleep(0.05)

threading.Thread(target=sync_produce, daemon=True).start()

# Async consumer — use in FastAPI route handlers, aiohttp, etc.
async def consume_async():
    tokens = []
    # 'async for token in RelayConsumer(...)' works directly (via __aiter__)
    async for token in RelayConsumer(relay_url, ch6):
        tokens.append(token)
        print(f"  received: {token!r}")
    return "".join(tokens)

result = await consume_async()
print(f"Full: {result!r}")

### 4.2 FastAPI SSE endpoint pattern

The typical pattern in production: submit an HPC job, then stream its tokens as Server-Sent Events.

In [ ]:
# This is the pattern used inside hpc-as-api — shown here so you understand
# what's happening under the hood.

from fastapi import FastAPI
from fastapi.responses import StreamingResponse

example_app = FastAPI()

@example_app.post("/run")
async def run_hpc_job(request: dict):
    """
    Submit an HPC job and stream its output as SSE.

    In production, 'submit_to_globus_compute()' would call Globus Compute
    and return a channel_id. Here we simulate it with a background thread.
    """
    channel_id = str(uuid.uuid4())

    # Submit job (non-blocking — returns immediately)
    # In production: future = gc_executor.submit(my_fn, relay_url=relay_url, channel_id=channel_id)
    def fake_hpc_job():
        with RelayProducer(relay_url, channel_id) as relay:
            for word in f"processing {request.get('data', 'input')}".split():
                relay.send_token(word + " ")
                time.sleep(0.1)

    threading.Thread(target=fake_hpc_job, daemon=True).start()

    # Stream relay output as SSE to the HTTP client
    async def sse_generator():
        async for token in RelayConsumer(relay_url, channel_id):
            yield f"data: {token}\n\n"
        yield "data: [DONE]\n\n"

    return StreamingResponse(sse_generator(), media_type="text/event-stream")

print("FastAPI SSE pattern defined (not running — requires uvicorn to serve).")

---
## Part 5: End-to-end encryption (AES-256-GCM)

TLS (wss://) protects data in transit, but the relay server itself sees plaintext. For sensitive data (medical records, PII, financial), encrypt payloads so the relay forwards opaque ciphertext.

### How it works

```
Producer  →  encrypt(token)  →  relay  →  decrypt(ciphertext)  →  Consumer
               AES-256-GCM                     AES-256-GCM
```

The relay sees `{"type":"enc","d":"<opaque base64 blob>"}` — it cannot read the content.

In [ ]:
from streamrelay import generate_key, encrypt_message, decrypt_message

# Generate a 32-byte AES-256 key as 64 hex chars
# Generate ONCE, store in env vars on BOTH producer (HPC) and consumer sides
key = generate_key()
print(f"Key (first 16 chars shown): {key[:16]}...")
print(f"Key length: {len(key)} hex chars = {len(key)//2} bytes")

# Manual encryption/decryption
plaintext = '{"type": "token", "content": "patient_id=12345 has diagnosis X"}'
encrypted = encrypt_message(key, plaintext)
print(f"\nEncrypted: {encrypted[:60]}...")

decrypted = decrypt_message(key, encrypted)
print(f"Decrypted: {decrypted}")

In [ ]:
# Full encrypt/decrypt roundtrip with producer/consumer
ch7 = str(uuid.uuid4())

secret_message = "CLASSIFIED: experiment results showing 94% accuracy"

def produce_encrypted():
    # encryption_key= encrypts every message before sending to relay
    with RelayProducer(relay_url, ch7, encryption_key=key) as relay:
        for word in secret_message.split():
            relay.send_token(word + " ")
            time.sleep(0.03)

threading.Thread(target=produce_encrypted, daemon=True).start()

# Consumer decrypts with the same key
result = RelayConsumer(relay_url, ch7, encryption_key=key).collect()
print(f"Decrypted message: {result!r}")

# What the relay server actually sees (encrypted blob)
ch8 = str(uuid.uuid4())

def produce_for_sniff():
    with RelayProducer(relay_url, ch8, encryption_key=key) as relay:
        relay.send_token("secret")

threading.Thread(target=produce_for_sniff, daemon=True).start()

# Simulate a relay operator reading the raw WebSocket traffic
with ws_connect(f"{relay_url}/consume/{ch8}") as ws:
    raw = ws.recv()
    msg = json.loads(raw)
    print(f"\nWhat the relay sees: type={msg['type']}, data={msg.get('d', '')[:40]}...")
    print("The relay cannot read the plaintext!")

### Key generation and distribution

```bash
# Generate once:
python -c "from streamrelay import generate_key; print(generate_key())"

# Add to producer side (Globus endpoint config.yaml or environment):
RELAY_ENCRYPTION_KEY=<64-hex-char output>

# Add to consumer side (proxy EnvironmentFile):
RELAY_ENCRYPTION_KEY=<same 64-hex-char output>
```

The key must be **the same** on both sides. It never travels over the relay.

---
## Part 6: StreamingExecutor — Globus Compute integration

`StreamingExecutor` is a high-level class that wraps Globus Compute job submission and relay consumption into a single `async for` loop.

### 6.1 What it does

```python
async with StreamingExecutor(endpoint_id, relay_url, secret) as executor:
    async for token in executor.stream(my_fn, arg1=val1, arg2=val2):
        print(token, end="")
```

Under the hood:
1. Generates a random `channel_id`
2. Submits `my_fn(arg1=val1, arg2=val2, relay_url=..., channel_id=...)` to Globus Compute (non-blocking)
3. Connects to relay as consumer
4. Yields tokens as they arrive
5. After stream ends, checks for Globus-level errors

### 6.2 Your HPC function must use RelayProducer

In [ ]:
# This is what your Globus Compute function looks like:

def my_hpc_function(prompt: str, relay_url: str, channel_id: str,
                    relay_secret: str = "", encryption_key: str = ""):
    """
    Runs on the HPC cluster via Globus Compute.

    The four extra kwargs (relay_url, channel_id, relay_secret, encryption_key)
    are injected automatically by StreamingExecutor — you don't pass them explicitly.
    """
    # All imports must be INSIDE the function for Globus Compute serialization
    from streamrelay import RelayProducer

    with RelayProducer(relay_url, channel_id,
                       relay_secret=relay_secret,
                       encryption_key=encryption_key) as relay:
        # Replace this with your actual HPC computation:
        import time
        for word in f"Processing prompt: {prompt}".split():
            relay.send_token(word + " ")
            time.sleep(0.05)

print("HPC function defined.")

In [ ]:
# Using StreamingExecutor (requires a live Globus endpoint)
# Shown as comments — run against your real endpoint

# from streamrelay import StreamingExecutor
#
# async with StreamingExecutor(
#     endpoint_id="8d978809-eec4-413d-bbd4-b099e488100a",
#     relay_url="wss://relay.stream.acer.uic.edu",
#     relay_secret="your-relay-secret",
#     encryption_key="your-64-hex-key",   # optional
# ) as executor:
#     async for token in executor.stream(
#         my_hpc_function,
#         prompt="What is the Schrödinger equation?"
#         # relay_url, channel_id, relay_secret injected automatically
#     ):
#         print(token, end="", flush=True)

print("(Live Globus required — see commented code above)")

### 6.3 The exec()-from-source trick for PyInstaller

When using hpc-as-api with a PyInstaller-bundled desktop app, normal function serialization breaks because Globus Compute captures PyInstaller's internal bytecode references. The workaround: define the remote function from a **source string** at runtime.

```python
_SOURCE = '''
def my_fn(prompt, relay_url, channel_id, relay_secret=""):
    from streamrelay import RelayProducer
    with RelayProducer(relay_url, channel_id, relay_secret=relay_secret) as relay:
        relay.send_token("hello")
'''
_ns = {}
exec(compile(_SOURCE, "<my_fn>", "exec"), _ns)
my_fn = _ns["my_fn"]
# Now pass my_fn to StreamingExecutor.stream() or gc_executor.submit()
```

This produces clean bytecode with no bundler references — works in both normal Python and PyInstaller environments.

---
## Part 7: Production deployment

### 7.1 The lifecycle CLI

streamrelay 0.3.0 adds a full lifecycle CLI:

In [ ]:
import subprocess
result = subprocess.run([sys.executable, "-m", "streamrelay.cli", "--help"],
                        capture_output=True, text=True)
print(result.stdout)

### 7.2 Deploying to a Linux VM (systemd)

```bash
# On the relay VM (Ubuntu/Debian/RHEL):

# 1. Install
pip install streamrelay

# 2. Install as systemd service
sudo streamrelay install \
    --port 8765 \
    --secret "$(python3 -c 'import secrets; print(secrets.token_hex(32))')" \
    --max-buffer 500 \
    --channel-timeout 300

# 3. Start
sudo streamrelay start

# 4. Check status + WebSocket health
streamrelay status

# 5. Restart after config changes
sudo streamrelay restart

# 6. Remove
sudo streamrelay uninstall
```

The generated `/etc/systemd/system/streamrelay.service` looks like:

```ini
[Unit]
Description=streamrelay WebSocket Relay Server
After=network.target

[Service]
User=ubuntu
Environment=RELAY_SECRET=your-secret
ExecStart=/usr/bin/python3.11 -m streamrelay.server --host 0.0.0.0 --port 8765 --max-buffer 500 --channel-timeout 300 --log-level INFO
Restart=always
RestartSec=5

[Install]
WantedBy=multi-user.target
```

### 7.3 TLS (wss://) with Caddy

The relay binds to `0.0.0.0:8765`. Put Caddy in front for TLS:

```
# /etc/caddy/Caddyfile
relay.example.com {
    reverse_proxy localhost:8765
}
```

```bash
sudo systemctl reload caddy
# Relay is now at wss://relay.example.com
```

Caddy auto-provisions a free TLS certificate from Let's Encrypt.

### 7.4 Security checklist for production

| Item | Command/Setting |
|------|----------------|
| ✅ Shared secret | `--secret` or `RELAY_SECRET=...` |
| ✅ TLS (wss://) | Caddy reverse proxy |
| ✅ E2E encryption | `RELAY_ENCRYPTION_KEY=` on both ends |
| ✅ Firewall | Allow only port 443 (Caddy) + your Caddy→relay connection |
| ✅ Auto-restart | `Restart=always` in systemd unit |
| ✅ Start on boot | `sudo systemctl enable streamrelay` (done by `install`) |

### 7.5 Monitoring and health

```bash
# Quick health check
streamrelay status --host relay.example.com --port 8765

# Manual WebSocket health check (useful from scripts)
python3 -c "
from websockets.sync.client import connect
import json
with connect('wss://relay.example.com/health') as ws:
    print(json.loads(ws.recv()))
"

# View live logs (systemd)
sudo journalctl -u streamrelay -f
```

---
## Summary

```python
# The three-line pattern:

# 1. On HPC (producer side):
with RelayProducer(relay_url, channel_id, relay_secret=secret) as relay:
    relay.send_token(output)

# 2. On your app (consumer side, sync):
for token in RelayConsumer(relay_url, channel_id, relay_secret=secret).stream():
    print(token)

# 3. On your app (consumer side, async / FastAPI):
async for token in RelayConsumer(relay_url, channel_id):
    yield f"data: {token}\n\n"
```

| Class | Use when |
|-------|----------|
| `RelayProducer` | Your code runs on HPC and generates output |
| `RelayConsumer` | Your app receives output from HPC |
| `StreamingExecutor` | You want a one-liner for Globus Compute + relay |
| `start_relay()` | You need to embed the relay in a Python process |
| `streamrelay serve` | Start the relay from the command line |
| `streamrelay install` | Set up as a systemd/launchd service |

## Next steps

1. Deploy a relay VM: `sudo streamrelay install --secret your-secret`
2. Test with `streamrelay status`
3. Use `RelayProducer` in your Globus Compute function
4. Use `RelayConsumer` or `StreamingExecutor` in your app
5. Enable encryption with `generate_key()` + `RELAY_ENCRYPTION_KEY` env var

In [ ]:
# Cleanup
relay_proc.terminate()
relay_proc.wait()
print("Local relay stopped. Tutorial complete!")